In [1]:
import os
os.environ['R_HOME'] = '/home/yanxh/anaconda3/envs/R4.3/lib/R'
os.environ['R_USER'] = '/home/yanxh/anaconda3/envs/R4.3/lib/python3.8/site-packages/rpy2'
import psutil

# Get the current process
# p = psutil.Process(os.getpid())

# # Set the process to run only on the first CPU core
# p.cpu_affinity([5])

In [2]:
import os
import pandas as pd
import pickle
import matplotlib.pyplot as plt

import anndata as ad
import numpy as np
import yaml
import sys
import logging
import scanpy as sc
from os.path import join
import scipy.sparse as sps
from sklearn.cluster import KMeans

import scglue

import warnings
def wrap_warn_plot(adata, basis, color, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        sc.pl.embedding(adata, basis=basis, color=color, **kwargs)

def get_umap(ad, use_reps=[]):
    for use_rep in use_reps:
        umap_add_key = f'{use_rep}_umap'
        sc.pp.neighbors(ad, use_rep=use_rep, n_neighbors=15)
        sc.tl.umap(ad)
        ad.obsm[umap_add_key] = ad.obsm['X_umap']
    return ad

from sklearn.metrics import adjusted_rand_score

def split_ob(ads, ad_ref, ob='obs', key='emb2'):
    len_ads = [_.n_obs for _ in ads]
    if ob=='obsm':
        split_obsms = np.split(ad_ref.obsm[key], np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obsms):
            ad.obsm[key] = v
    else:
        split_obs = np.split(ad_ref.obs[key].to_list(), np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obs):
            ad.obs[key] = v

In [10]:
root_data_dir = '../../../data/processed/Simulations/'

omics = "cite"
# latent_dim = 20  # search
x2u_h_depth = 2
x2u_h_dim = 512 
u2x_h_depth = 1
# u2x_h_dim = 128  # the same as du_h_dim
du_h_depth = 2
# du_h_dim = 128  # search
dropout = 0.2
lam_data = 1.0
lam_kl = 1.0  
lam_align = 2.0   
lam_cross = 2.0 
lam_cos = 1.0 
normalize_u = True
random_seed = 5

latent_dim = 50
u2x_h_dim = 256
du_h_dim  = 256
du_h_depth = 2
max_epochs1 = 80
max_epochs2 = 50

for zero_p in ['0.1', '0.2', '0.3', '0.4', '0.5']:  # ['0.1', '0.2', '0.3', '0.4', '0.5']
    data_dirs = os.listdir(root_data_dir)
    foldn = list(filter(lambda x: x.startswith(f'rna-{zero_p}'), data_dirs))[0]
    data_dir = root_data_dir + foldn

    for seed in [1, 2, 3]:
        out_dir = join(root_out_dir, f'zero_p={zero_p}-seed={seed}')
        os.makedirs(out_dir, exist_ok=True)

        input_train_mod1 = sc.read_h5ad(join(data_dir, f'{seed}/ad_rna1.h5ad'))
        input_train_mod2 = sc.read_h5ad(join(data_dir, f'{seed}/ad_adt1.h5ad'))
        input_test_mod1 = sc.read_h5ad(join(data_dir, f'{seed}/ad_rna2.h5ad'))
        input_test_mod2 = sc.read_h5ad(join(data_dir, f'{seed}/ad_adt2.h5ad'))

        input_train_mod1.X = sps.csr_matrix(input_train_mod1.layers['counts'])
        input_train_mod2.X = sps.csr_matrix(input_train_mod2.layers['counts'])
        input_test_mod1.X = sps.csr_matrix(input_test_mod1.layers['counts'])
        input_test_mod2.X = sps.csr_matrix(input_test_mod2.layers['counts'])

        input_train_mod1.obs["uid"] = [f"train-{i}" for i in range(input_train_mod1.shape[0])]
        input_train_mod2.obs["uid"] = [f"train-{i}" for i in range(input_train_mod2.shape[0])]
        input_test_mod1.obs["uid"] = [f"test-mod1-{i}" for i in range(input_test_mod1.shape[0])]
        input_test_mod2.obs["uid"] = [f"test-mod2-{i}" for i in range(input_test_mod2.shape[0])]
        input_test_mod1.obs_names = [f"test-mod1-{i}" for i in range(input_test_mod1.shape[0])]
        input_test_mod2.obs_names = [f"test-mod2-{i}" for i in range(input_test_mod2.shape[0])]

        input_train_mod1.obs['batch'] = 's1'
        input_train_mod2.obs['batch'] = 's1'
        input_test_mod1.obs['batch'] = 's2'
        input_test_mod2.obs['batch'] = 's3'

        print('Reading `h5ad` files...')
        omics = "cite"

        ### preprocessing
        sys.path.insert(0, './resources')
        import utils

        gex = input_train_mod1
        other = input_train_mod2
        n_genes = min(5000, input_train_mod1.shape[1])

        print('Preprocessing GEX...')
        gex_prep = utils.GEXPreprocessing(n_comps=100, n_genes=n_genes, merge_adt=omics == "cite")
        gex_prep.fit_transform(gex)
        print('Preprocessing ADT...')
        other_prep = utils.ADTPreprocessing(n_comps=min(100, input_train_mod2.shape[1]))
        other_prep.fit_transform(other)

        with open(os.path.join(out_dir, "prep.pickle"), "wb") as f:
            pickle.dump({
                "gex_prep": gex_prep,
                "other_prep": other_prep
            }, f)

        with open(f"{out_dir}/hyperparams.yaml", "w") as f:
            yaml.dump({
                "n_genes": n_genes,
                "latent_dim": latent_dim,
                "x2u_h_depth": x2u_h_depth,
                "x2u_h_dim": x2u_h_dim,
                "u2x_h_depth": u2x_h_depth,
                "u2x_h_dim": u2x_h_dim,
                "du_h_depth": du_h_depth,
                "du_h_dim": du_h_dim,
                "dropout": dropout,
                "lam_data": lam_data,
                "lam_kl": lam_kl,
                "lam_align": lam_align,
                "lam_cross": lam_cross,
                "lam_cos": lam_cos,
                "normalize_u": normalize_u,
                "random_seed": random_seed
            }, f)

        scglue.models.configure_dataset(
            gex, "NB", use_highly_variable=True,
            use_layer="counts", use_rep="X_pca",
            use_batch="batch", use_uid="uid"
        )
        scglue.models.configure_dataset(
            other, "NB", use_highly_variable=True,
            use_layer="counts", use_rep="X_pca",
            use_batch="batch", use_uid="uid"
        )

        print('Building model...')
        model = scglue.models.SCCLUEModel(
            {"gex": gex, "other": other},
            latent_dim=latent_dim,
            x2u_h_depth=x2u_h_depth,
            x2u_h_dim=x2u_h_dim,
            u2x_h_depth=u2x_h_depth,
            u2x_h_dim=u2x_h_dim,
            du_h_depth=du_h_depth,
            du_h_dim=du_h_dim,
            dropout=dropout,
            shared_batches=False,
            random_seed=random_seed
        )

        print('Compiling model...')
        model.compile(
            lam_data=lam_data, lam_kl=lam_kl, lam_align=lam_align,
            lam_cross=lam_cross, lam_cos=lam_cos, normalize_u=normalize_u,
            domain_weight={"gex": 1, "other": 1}
        )

        # help(model.fit)
        print('Training model...')
        model.fit(
            {"gex": gex, "other": other}, max_epochs=max_epochs1,
        )
        model.save(os.path.join(out_dir, "pretrain.dill"))

        # ===============================
        #               stage 2
        # ===============================

        logging.info('Concatenating training and test data...')
        input_mod1 = ad.concat(
            {"train": input_train_mod1, "test": input_test_mod1},
            axis=0, join="outer", merge="same", label="group",
            fill_value=0, index_unique="-"
        )
        input_mod1.uns["feature_type"] = 'GEX'

        input_mod2 = ad.concat(
            {"train": input_train_mod2, "test": input_test_mod2},
            axis=0, join="outer", merge="same", label="group",
            fill_value=0, index_unique="-"
        )
        input_mod2.uns["feature_type"] = 'ADT'

        gex, other = input_mod1, input_mod2

        logging.info('Reading preprocessors...')
        with open(os.path.join(
               out_dir, "prep.pickle"
        ), "rb") as f:
            prep = pickle.load(f)
            gex_prep = prep["gex_prep"]
            other_prep = prep["other_prep"]

        logging.info('Preprocessing...')
        gex_prep.transform(gex)
        other_prep.transform(other)

        logging.info('Fine-tuning model...')
        scglue.models.configure_dataset(
            gex, "NB", use_highly_variable=True,
            use_layer="counts", use_rep="X_pca",
            use_batch="batch", use_uid="uid"
        )
        scglue.models.configure_dataset(
            other, "NB", use_highly_variable=True,
            use_layer="counts", use_rep="X_pca",
            use_batch="batch", use_uid="uid"
        )

        with open(os.path.join(
                out_dir, "hyperparams.yaml"
        ), "r") as f:
            hyperparams = yaml.load(f, Loader=yaml.Loader)

        logging.info('Building model...')
        model = scglue.models.SCCLUEModel(
            {"gex": gex, "other": other},
            latent_dim=hyperparams["latent_dim"],
            x2u_h_depth=hyperparams["x2u_h_depth"],
            x2u_h_dim=hyperparams["x2u_h_dim"],
            u2x_h_depth=hyperparams["u2x_h_depth"],
            u2x_h_dim=hyperparams["u2x_h_dim"],
            du_h_depth=hyperparams["du_h_depth"],
            du_h_dim=hyperparams["du_h_dim"],
            dropout=hyperparams["dropout"],
            shared_batches=False,
            random_seed=hyperparams["random_seed"]
        )
        # print(model.net)

        logging.info('Adopting pretrained weights...')
        model.adopt_pretrained_model(scglue.models.load_model(os.path.join(
            out_dir, "pretrain.dill"
        )))

        logging.info('Compiling model...')
        model.compile(
            lam_data=hyperparams["lam_data"],
            lam_kl=hyperparams["lam_kl"],
            lam_align=hyperparams["lam_align"],
            lam_cross=hyperparams["lam_cross"],
            lam_cos=hyperparams["lam_cos"],
            normalize_u=hyperparams["normalize_u"],
            domain_weight={"gex": 1, "other": 1},
            lr=1e-3  # TODO: Fine-tuning learning rate
        )

        logging.info('Training model...')
        model.fit(
            {"gex": gex, "other": other},
            align_burnin=0, max_epochs=max_epochs2,
            patience=8, reduce_lr_patience=3
        )

        logging.info('Projecting cell embeddings...')
        gex.obsm["X_model"] = model.encode_data("gex", gex)
        other.obsm["X_model"] = model.encode_data("other", other)

        input_train_mod1.obsm['emb'] = gex.obsm["X_model"][:input_train_mod1.n_obs]
        input_test_mod1.obsm['emb'] = gex.obsm["X_model"][input_train_mod1.n_obs:]

        input_train_mod2.obsm['emb'] = other.obsm["X_model"][:input_train_mod2.n_obs]
        input_test_mod2.obsm['emb'] = other.obsm["X_model"][input_train_mod2.n_obs:]

        ad_train_mean_enc = sc.AnnData(
            (input_train_mod1.obsm['emb'] + input_train_mod2.obsm['emb'])/2, 
            obs=input_train_mod1.obs.copy()
        )
        ad_train_mean_enc.obsm['emb'] = ad_train_mean_enc.X.copy()
        ad_train_mean_enc.obsm['spatial'] = input_train_mod1.obsm['spatial']
        ad_mosaic = sc.concat([ad_train_mean_enc, input_test_mod1, input_test_mod2])

        df_emb = pd.DataFrame(ad_mosaic.obsm['emb'], 
                index=input_train_mod1.obs_names.to_list() + input_test_mod1.obs_names.to_list() + input_test_mod2.obs_names.to_list())
        
        os.makedirs(f'../../../results/embeddings/CLUE/Simulations/{foldn}/{seed}', exist_ok=True)
        df_emb.to_csv(f'../../../results/embeddings/CLUE/Simulations/{foldn}/{seed}/df_emb.csv')
       

Reading `h5ad` files...
Preprocessing GEX...
Preprocessing ADT...
Building model...
Compiling model...
Training model...
[INFO] SCCLUEModel: Setting `align_burnin` = 659


INFO:SCCLUEModel:Setting `align_burnin` = 659


[INFO] SCCLUEModel: Setting `patience` = 494


INFO:SCCLUEModel:Setting `patience` = 494


[INFO] SCCLUEModel: Setting `reduce_lr_patience` = 165


...

INFO:Transferrable:Copied: encoders.1.bn_1.running_var


[INFO] Transferrable: Copied: encoders.1.bn_1.num_batches_tracked


INFO:Transferrable:Copied: encoders.1.bn_1.num_batches_tracked


[INFO] BatchedTransferrable: Copied: scale_lin


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

